# AMEX Enterprise Credit Risk Platform
## Notebook 50 -- Collections Optimization: Business Understanding & Policy
### Phase 4 . Problem Statement 9: Collections Optimization

CRISP-DM stage: **Business Understanding**. Depends on Problem 1 Notebooks 01/02/05/08, Problem 4's real
severity bundle (Notebook 28), and Problem 8's real, production-recommended transition matrix (Notebook 48).

**What this notebook does:** defines the real business problem (per-customer propensity-to-cure scoring,
built on Problem 8's already-validated population-level transition matrix, plus a treatment-assignment
policy), sets the cure label definition and KPI target, defines an honest business-rule treatment-tier
policy (explicitly NOT a fitted treatment-response model -- this dataset has no real collections-contact
history to fit one against), and writes `collections_policy.json` for Notebook 51 (Modeling) to consume.

**HYPER note:** built from the same master notebook template this platform already established (Notebook 46,
Problem 8's Business Understanding notebook) -- Sections 1-3 and 11-12's structure are reused verbatim where
the logic is genuinely identical, per this project's own template-reuse convention.

**WARP note:** Section 2 tightens this platform's resource cap to 92% CPU / 92% RAM (down from the original
95%/90% split) for this notebook and every Phase 4 notebook after it, following the real Phase 3 hang
incident and the user's explicit 2026-08-25 directive -- see Section 2's inline rationale.

Zero-fabrication statement: every number this notebook prints is either computed live against the real raw
Kaggle CSVs, or an explicitly labeled ASSUMPTION -- no results are hardcoded or estimated in advance.
**Real bug found and fixed (2026-08-25, reported by the user running this notebook on their own machine against real, already-completed Notebook 48 output):** Section 1 previously guessed at two candidate paths for `roll_rate_deployment_policy.json` (a `docs/` folder nested under Problem 8, and the flat `PROJECT_ROOT/artifacts/` folder). Neither guess is where Notebook 48 actually writes the file -- it writes to `RR_DEPLOYMENT_DIR / "policy_artifacts" / "roll_rate_deployment_policy.json"`, a third location neither candidate covered, so this notebook raised a real `FileNotFoundError` against a genuinely completed Notebook 48 run. Fixed by reading the path Notebook 48 itself recorded in `notebook_48_summary.json`'s `deployment_policy_path` key, instead of re-deriving/guessing the folder layout a second time -- the same canonical-source-of-truth pattern this notebook already used for Notebook 28's severity bundle.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05,
#            08), PROBLEM 4'S REAL SEVERITY BUNDLE (NOTEBOOK 28), AND PROBLEM
#            8'S REAL, PRODUCTION-RECOMMENDED TRANSITION MATRIX (NOTEBOOK 48)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 08, 28, 48")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB48_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_48_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first (real EAD/LGD assumptions inherited, "
                         "not re-guessed)"),
    (NB48_SUMMARY_PATH, "run 48_roll_rate_modeling_validation_deployment.ipynb (Problem 8) first -- "
                         "Problem 9 depends on Problem 8's real, production-recommended transition "
                         "matrix per the master plan."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB48_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB48_SUMMARY = json.load(f)

# --- Problem 4's real severity-scoring bundle -- same nested-then-flat
#     resolution this platform's own Notebook 46 (Problem 8) already had to
#     add for real, because Problem 4's notebooks (26-29) are the one place
#     that write their summary JSON to a per-problem nested artifacts folder
#     rather than the flat PROJECT_ROOT/artifacts/ folder every other problem
#     uses. Reusing that exact fallback here rather than re-deriving it --
#     the same real path bug would otherwise resurface for a second problem. ---
P4_ROOT = (
    PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning"
    / "Problem4_Delinquency_Escalation_Loss_Severity"
)
NB28_SUMMARY_PATH_NESTED = P4_ROOT / "artifacts" / "notebook_28_summary.json"
NB28_SUMMARY_PATH_FLAT = ARTIFACTS_DIR / "notebook_28_summary.json"
if NB28_SUMMARY_PATH_NESTED.exists():
    NB28_SUMMARY_PATH = NB28_SUMMARY_PATH_NESTED
elif NB28_SUMMARY_PATH_FLAT.exists():
    NB28_SUMMARY_PATH = NB28_SUMMARY_PATH_FLAT
else:
    raise FileNotFoundError(
        f"notebook_28_summary.json not found at either its expected nested location "
        f"({NB28_SUMMARY_PATH_NESTED}) or the flat fallback location ({NB28_SUMMARY_PATH_FLAT}).\n"
        f"Fix: run 28_validation_deployment.ipynb (Problem 4) first -- Problem 9 depends on "
        f"Problem 4 per the master plan, inheriting its real EAD/LGD-adjacent severity context."
    )
with open(NB28_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB28_SUMMARY = json.load(f)
if "severity_scoring_bundle.json" not in NB28_SUMMARY.get("output_files", {}):
    raise KeyError("notebook_28_summary.json has no 'severity_scoring_bundle.json' entry under output_files.")
SEVERITY_BUNDLE_PATH = Path(NB28_SUMMARY["output_files"]["severity_scoring_bundle.json"])
if not SEVERITY_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"{SEVERITY_BUNDLE_PATH} not found.\nFix: re-run Notebook 28 (Problem 4).")
with open(SEVERITY_BUNDLE_PATH, "r", encoding="utf-8") as f:
    P4_SEVERITY_BUNDLE = json.load(f)
P4_TIER_ORDER = P4_SEVERITY_BUNDLE["tier_order"]

# --- Problem 8's real, RECOMMENDED-FOR-PRODUCTION Markov transition matrix
#     (Notebook 48's deployment policy) -- the actual source of truth Problem
#     9 builds on.
#
#     REAL BUG, CAUGHT ON THE USER'S OWN MACHINE (not in this sandbox --
#     no raw data here to execute against) AND FIXED HERE: this section
#     previously guessed at two candidate paths for
#     roll_rate_deployment_policy.json (a "docs" folder nested under
#     Problem 8, and the flat PROJECT_ROOT/artifacts/ folder). Neither
#     guess is where Notebook 48 actually writes the file -- Notebook 48's
#     own Section 1 defines RR_DEPLOYMENT_DIR (from pillar_dirs, or a
#     "deployment" folder fallback) and then writes the policy to
#     RR_DEPLOYMENT_DIR / "policy_artifacts" / "roll_rate_deployment_policy.json",
#     a THIRD location neither candidate here covered -- so this notebook
#     raised a real FileNotFoundError against a live, already-completed
#     Notebook 48 run. Fixed by reading the path Notebook 48 itself
#     recorded in its own summary JSON (notebook_48_summary.json's
#     "deployment_policy_path" key) instead of re-deriving/guessing the
#     folder layout a second time -- the same canonical-source-of-truth
#     pattern this notebook already uses for Notebook 28's severity bundle
#     (via notebook_28_summary.json's output_files) and every other
#     cross-problem dependency in this platform. ---
NB48_DEPLOYMENT_POLICY_PATH_STR = NB48_SUMMARY.get("deployment_policy_path")
if not NB48_DEPLOYMENT_POLICY_PATH_STR:
    raise KeyError(
        "notebook_48_summary.json has no 'deployment_policy_path' entry.\n"
        "Fix: re-run 48_roll_rate_modeling_validation_deployment.ipynb (Problem 8)."
    )
P8_DEPLOYMENT_POLICY_PATH = Path(NB48_DEPLOYMENT_POLICY_PATH_STR)
if not P8_DEPLOYMENT_POLICY_PATH.exists():
    raise FileNotFoundError(
        f"roll_rate_deployment_policy.json not found at the location recorded in "
        f"notebook_48_summary.json ({P8_DEPLOYMENT_POLICY_PATH}).\n"
        f"Fix: re-run 48_roll_rate_modeling_validation_deployment.ipynb (Problem 8) -- Problem 9 "
        f"depends on Problem 8's real, production-recommended transition matrix per the master plan."
    )
with open(P8_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P8_DEPLOYMENT_POLICY = json.load(f)

P8_STATE_NAMES = P8_DEPLOYMENT_POLICY["state_names"]
P8_TRANSITION_MATRIX = P8_DEPLOYMENT_POLICY["transition_matrix"]
P8_MONITORED_FEATURES = P8_DEPLOYMENT_POLICY["monitored_features"]
P8_FEATURE_WEIGHTS = P8_DEPLOYMENT_POLICY["feature_weights"]
P8_CUT_LOW = P8_DEPLOYMENT_POLICY["cut_low"]
P8_CUT_HIGH = P8_DEPLOYMENT_POLICY["cut_high"]
P8_RECOMMENDED_FOR_PRODUCTION = P8_DEPLOYMENT_POLICY["recommended_for_production"]
if not P8_RECOMMENDED_FOR_PRODUCTION:
    print(
        "WARNING: Problem 8's roll-rate model is NOT currently recommended for production. Problem 9 "
        "still reuses its real, measured transition matrix (the matrix itself is real regardless of the "
        "recommendation flag), but this is noted honestly rather than silently assumed away."
    )

EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "collections_policy" in PILLAR_DIRS:
    COLLECTIONS_POLICY_DIR = PILLAR_DIRS["collections_policy"]
else:
    COLLECTIONS_POLICY_DIR = (
        PROJECT_ROOT / "Phase4_Operational_Risk_Management"
        / "Problem9_Collections_Optimization" / "policy"
    )
    print(f"NOTE: 'collections_policy' not in pillar_dirs -- using fallback: {COLLECTIONS_POLICY_DIR}")
COLLECTIONS_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                         : {CONFIG_PATH}")
print(f"Reused Problem 4's real severity bundle    : {SEVERITY_BUNDLE_PATH}")
print(f"Reused Problem 8's real deployment policy  : {P8_DEPLOYMENT_POLICY_PATH}")
print(f"Problem 8 state names (reused verbatim)    : {P8_STATE_NAMES}")
print(f"Problem 8 recommended for production       : {P8_RECOMMENDED_FOR_PRODUCTION}")
print(f"Champion architecture (Problem 1, measured) : {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"EAD/LGD (Notebook 08, inherited)            : ${EAD_PER_ACCOUNT_USD:,} / {LGD_ASSUMPTION:.0%}")
print(f"Policy artifacts will be written under: {COLLECTIONS_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 4 TIGHTENED
#            CAP -- 92% CPU / 92% RAM, DOWN FROM THE PLATFORM'S ORIGINAL
#            95% CPU / 90% RAM SPLIT)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports (Phase 4 Tightened Cap)")

# --- Standing correction (user directive, 2026-08-25): the platform's
#     original resource_limits (95% CPU / 90% RAM, see project_config.json)
#     were in effect during the real hanging incident encountered in Phase 3.
#     A hang is not conclusively proven to trace to the cap alone, but it is
#     a plausible contributing factor (a near-saturated CPU with the OS still
#     needing headroom for its own scheduler and any memory-pressure
#     response), so from this notebook forward the platform's own historical
#     caps are TIGHTENED to a uniform 92% for both CPU threads and RAM,
#     layered on top of (never above) whatever the historical config already
#     computed -- this does not silently rewrite project_config.json (a real,
#     measured artifact from the user's actual hardware detection run); it
#     computes a stricter derived ceiling for this notebook and every Phase 4
#     notebook after it. ---
_PHASE4_CPU_FRACTION_CAP = 0.92
_PHASE4_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(
    _historical_thread_count,
    max(1, round(DETECTED_LOGICAL_CORES * _PHASE4_CPU_FRACTION_CAP)),
)
MAX_RAM_BYTES = min(
    _historical_max_ram_bytes,
    round(DETECTED_TOTAL_RAM_BYTES * _PHASE4_RAM_FRACTION_CAP),
)

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(
    f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads "
    f"({WARP_THREAD_COUNT / DETECTED_LOGICAL_CORES:.0%}, Phase 4 tightened cap, min of historical "
    f"{_historical_thread_count} and 92% of detected cores)"
)
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling (Phase 4 tightened cap): {MAX_RAM_BYTES / 1e9:.1f} GB "
      f"(min of historical {_historical_max_ram_bytes / 1e9:.1f} GB and 92% of detected total)")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA CHECK -- CONFIRM PROBLEM 8'S REAL MONITORED COLUMNS
#            ARE STILL PRESENT (NO FRESH FEATURE SELECTION -- REUSE VERBATIM)
# =============================================================================
_section("SECTION 4: Live Schema Check -- Problem 8's Real Monitored Columns")

# --- Problem 9 does NOT fit a fresh severity-state scorer. It reuses Problem
#     8's real, already-fitted, production-recommended scorer (weights,
#     directions, means, stds, cut_low/cut_high) VERBATIM -- refitting here
#     would silently create a second, divergent state definition for the same
#     underlying concept, which is exactly the kind of drift this platform's
#     own Notebook 46 (Section 4) already flagged as a real risk between
#     Problems 4/6/8. Collections Optimization's job is to build ON TOP of
#     the validated state/transition definition, not redefine it. ---
with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)
_missing_cols = set(P8_MONITORED_FEATURES) - _header_cols
if _missing_cols:
    raise RuntimeError(
        f"{len(_missing_cols)} of Problem 8's real monitored column(s) are not present in the real raw "
        f"CSV header: {sorted(_missing_cols)}\nFix: investigate before proceeding -- this would mean the "
        f"raw file has changed since Problem 8 was built."
    )
print(f"Reused Problem 8's real monitored feature universe: {len(P8_MONITORED_FEATURES)} base columns "
      f"(verbatim -- confirmed present in the real raw CSV header)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION (SAME MEASURE
#            NOTEBOOK 46 ALREADY ESTABLISHED -- REUSED FOR ELIGIBILITY)
# =============================================================================
_section("SECTION 5: Real Per-Customer Statement-Count Distribution")

print("Reading real per-statement (raw, pre-aggregation) data from: " + str(RAW_TRAIN_DATA_PATH))
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
STATEMENT_COUNT_STATS = {
    "n_customers": _n_customers,
    "min": int(_counts_series.min()),
    "p10": float(_counts_series.quantile(0.10)),
    "p25": float(_counts_series.quantile(0.25)),
    "median": float(_counts_series.median()),
    "mean": float(_counts_series.mean()),
    "max": int(_counts_series.max()),
}
for _k in ("min", "p10", "p25", "median", "mean", "max"):
    print(f"  {_k:>6} statements/customer: {STATEMENT_COUNT_STATS[_k]}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BUSINESS UNDERSTANDING -- COLLECTIONS OPTIMIZATION (PROPENSITY-
#            TO-CURE SCORING + TREATMENT ASSIGNMENT)
# =============================================================================
_section("SECTION 6: Business Understanding -- Collections Optimization")

print(
    "PROBLEM 9 -- COLLECTIONS OPTIMIZATION (propensity-to-cure scoring + treatment assignment)\n\n"
    "Business case: Problem 8 answers 'what is the population-level probability a customer in state X "
    "moves to state Y next cycle'. Problem 9 asks the operational question a collections team actually "
    "needs answered: 'for THIS specific delinquent customer, right now, what is their individual "
    "probability of curing on their own, and does that customer need a call today or can they wait?' -- "
    "turning Problem 8's population-level transition matrix into a per-customer propensity score, ranked "
    "and tiered into a real treatment-assignment policy.\n\n"
    "CURE DEFINITION (reusing Problem 8's real, already-validated state machine, not a fresh one): a "
    "customer 'cures' between two consecutive statements if their fitted severity state IMPROVES -- "
    "Severe -> Moderate Severity, Severe -> Low Severity, or Moderate Severity -> Low Severity. This is "
    "the SAME state definition Problem 8 already fit and validated (recommended_for_production="
    f"{P8_RECOMMENDED_FOR_PRODUCTION}); Problem 9 does not invent a new one.\n\n"
    "WHY THIS IS A GENUINELY NEW MODEL, NOT A REPACKAGING OF PROBLEM 8: Problem 8's transition matrix is a "
    "POPULATION-LEVEL average -- it says 'X% of Moderate customers cure', but not WHICH ones. Notebook 51 "
    "trains a real per-customer classifier (same feature universe, same raw snapshot values Problem 8 "
    "already vetted) to predict cure probability for an INDIVIDUAL customer's real statement, which is "
    "exactly the ranking a collections team needs to decide who gets called first.\n\n"
    "DATA-LIMITATION HONESTY (same standing caveat as Problems 5/6/7/8): this dataset has NO real "
    "collections-contact history -- no record of which customers were ever called, emailed, offered a "
    "payment plan, or sent to an external agency, and no record of which treatment (if any) preceded any "
    "observed cure. This means the PROPENSITY-TO-CURE score is a real, measurable, trainable quantity "
    "(does this customer's own statement snapshot predict their next-state improvement), but TREATMENT "
    "ASSIGNMENT cannot be a fitted treatment-response model -- there is no real data on which treatment "
    "works best for which customer. Section 8 below defines treatment assignment honestly as a BUSINESS-"
    "RULE POLICY layer (propensity tier x dollar exposure), not a claim of a measured treatment effect."
)
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: CURE LABEL & ELIGIBILITY POLICY (ASSUMPTION)
# =============================================================================
_section("SECTION 7: Cure Label & Eligibility Policy (ASSUMPTION)")

# ASSUMPTION: eligibility population is customers observed in Moderate
# Severity or Severe at some statement with at least one following statement
# (so a real next-state outcome is observable) -- reusing Problem 8's real
# state names verbatim for continuity.
COLLECTIONS_ELIGIBLE_STATES = [s for s in P8_STATE_NAMES if s != "Low Severity"]

# ASSUMPTION: same true minimum Problem 8 already established -- a customer
# needs the statement itself (to score) AND the one immediately following it
# (to observe the real cure/no-cure outcome), so 2 real statements minimum.
MIN_STATEMENTS_FOR_CURE_LABEL = 2

# ASSUMPTION: a cure is a STRICT state improvement (moving to a state with a
# lower severity rank than the current one) -- partial improvement within
# the same state does not count, since Problem 8's own state machine has no
# finer resolution than the 3 named tiers to detect it.
CURE_DEFINITION = (
    "State improvement between two consecutive statements: current state in "
    f"{COLLECTIONS_ELIGIBLE_STATES}, next state strictly better (Severe -> Moderate Severity or Low "
    "Severity; Moderate Severity -> Low Severity)."
)

print(f"COLLECTIONS_ELIGIBLE_STATES (ASSUMPTION, reused from Problem 8): {COLLECTIONS_ELIGIBLE_STATES}")
print(f"MIN_STATEMENTS_FOR_CURE_LABEL (ASSUMPTION, true minimum for 1 observed outcome): "
      f"{MIN_STATEMENTS_FOR_CURE_LABEL}")
print(f"CURE_DEFINITION (ASSUMPTION): {CURE_DEFINITION}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: KPI TARGETS & TREATMENT-TIER POLICY -- HONEST, TECHNIQUE-
#            APPROPRIATE (ASSUMPTION)
# =============================================================================
_section("SECTION 8: KPI Targets & Treatment-Tier Policy (ASSUMPTION)")

# --- Predicting whether an ALREADY-delinquent customer improves at their
#     VERY NEXT statement, from a single snapshot, is a genuinely harder,
#     noisier task than Problem 1's whole-history default prediction (AUC
#     ~0.96) -- there is real short-term volatility in a single statement
#     that whole-history features average out. The KPI target below is set
#     modestly and honestly for that reason, not copied from Problem 1's
#     number. ---
COLLECTIONS_KPI_TARGETS = {
    "min_propensity_model_roc_auc": 0.60,
    "min_propensity_model_roc_auc_description": (
        "ASSUMPTION -- a deliberately modest, technique-appropriate bar (well below Problem 1's ~0.96 "
        "whole-history AUC): predicting a state improvement at the very next statement from a single "
        "snapshot is a harder, noisier task than whole-history default prediction. 0.60 is the minimum bar "
        "for 'this individual-level score carries real signal beyond the population base rate', not a "
        "claim of strong discrimination."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, 2026-08-25, carried from Problems 6/7/8): Notebook 51/52 must "
        "compute and DISPLAY -- inline in the notebook AND in this problem's Word/Excel/HTML reports -- "
        "the full classification metrics suite: ROC-AUC, PR-AUC, Accuracy, Precision, Recall, F1, "
        "Specificity, Log Loss, Matthews Correlation Coefficient, and a full confusion matrix."
    ),
    "treatment_tier_policy": {
        "description": (
            "ASSUMPTION, explicitly a business-rule policy layer, NOT a fitted treatment-response model "
            "(see Section 6's data-limitation honesty -- no real contact/treatment-outcome data exists in "
            "this dataset to fit one). Combines the real propensity-to-cure score (Notebook 51's measured "
            "output) with real severity-score magnitude (Problem 8's real per-statement composite score, "
            "reused verbatim -- NOT the flat EAD_PER_ACCOUNT_USD ASSUMPTION, which is a single constant "
            "identical for every account in this platform (confirmed in Problem 4's and Problem 8's own "
            "committed policy files) and so cannot differentiate anyone; caught during this notebook's own "
            "review, before it reached Notebook 51, and fixed to use a quantity that genuinely varies per "
            "customer) into 3 honest, operationally interpretable tiers."
        ),
        "tiers": [
            {
                "name": "Priority Outreach",
                "rule": "propensity-to-cure BELOW the population median AND severity score AT OR ABOVE "
                        "the population median (within the collections-eligible population)",
                "rationale": "Low self-cure chance + deeper into delinquency -- the customers a live agent "
                              "should call first, in severity-score-descending order within the tier.",
            },
            {
                "name": "Automated Nudge",
                "rule": "propensity-to-cure AT OR ABOVE the population median, any severity score",
                "rationale": "Likely to self-cure -- a lower-cost automated reminder (SMS/email) is "
                              "proportionate; live-agent time is not spent on customers who are likely to "
                              "resolve on their own.",
            },
            {
                "name": "Monitor",
                "rule": "propensity-to-cure BELOW the population median AND severity score BELOW the "
                        "population median",
                "rationale": "Low self-cure chance but comparatively less severe -- queued for standard-"
                              "cadence contact rather than priority outreach, given limited collections "
                              "capacity.",
            },
        ],
    },
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "problem_4_reference": {"tier_order": P4_TIER_ORDER},
    "problem_8_reference": {
        "state_names": P8_STATE_NAMES,
        "recommended_for_production": P8_RECOMMENDED_FOR_PRODUCTION,
        "transition_matrix": P8_TRANSITION_MATRIX,
    },
}
print(f"min_propensity_model_roc_auc (ASSUMPTION): {COLLECTIONS_KPI_TARGETS['min_propensity_model_roc_auc']}")
print(f"Treatment tiers (ASSUMPTION, business-rule policy, {len(COLLECTIONS_KPI_TARGETS['treatment_tier_policy']['tiers'])} total): "
      f"{[t['name'] for t in COLLECTIONS_KPI_TARGETS['treatment_tier_policy']['tiers']]}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: REAL CURE-LABEL ELIGIBILITY COVERAGE
# =============================================================================
_section("SECTION 9: Real Cure-Label Eligibility Coverage")

# --- Real, measured coverage: what fraction of the platform's customers have
#     enough statements (>=MIN_STATEMENTS_FOR_CURE_LABEL) to potentially
#     contribute a real cure/no-cure observation. This is an UPPER BOUND on
#     the training population -- Notebook 51 further restricts to statements
#     where the CURRENT state is Moderate/Severe, which requires actually
#     scoring every statement with Problem 8's real fitted formula (deferred
#     to Notebook 51, matching this platform's established phase split
#     between Business Understanding and Modeling). ---
_n_eligible_by_statement_count = int((_counts_series >= MIN_STATEMENTS_FOR_CURE_LABEL).sum())
CURE_LABEL_UPPER_BOUND_COVERAGE_PCT = 100.0 * _n_eligible_by_statement_count / _n_customers
print(f"Customers with >= {MIN_STATEMENTS_FOR_CURE_LABEL} statements (real, measured, upper bound on "
      f"cure-label eligibility): {_n_eligible_by_statement_count:,} / {_n_customers:,} "
      f"({CURE_LABEL_UPPER_BOUND_COVERAGE_PCT:.1f}%)")
print("The real, tighter figure (also requiring the CURRENT state to be Moderate/Severe) is a measured "
      "output of Notebook 51, not this notebook -- reported honestly as an upper bound here.")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE COLLECTIONS POLICY ARTIFACT
# =============================================================================
_section("SECTION 10: Write Collections Policy Artifact")

COLLECTIONS_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 9 -- Collections Optimization (Propensity-to-Cure Scoring + Treatment Assignment)",
    "collections_eligible_states": COLLECTIONS_ELIGIBLE_STATES,
    "min_statements_for_cure_label": MIN_STATEMENTS_FOR_CURE_LABEL,
    "cure_definition": CURE_DEFINITION,
    "cure_label_upper_bound_coverage_pct": CURE_LABEL_UPPER_BOUND_COVERAGE_PCT,
    "statement_count_stats": STATEMENT_COUNT_STATS,
    "reused_from_problem_8": {
        "deployment_policy_path": str(P8_DEPLOYMENT_POLICY_PATH),
        "state_names": P8_STATE_NAMES,
        "monitored_features": P8_MONITORED_FEATURES,
        "feature_weights": P8_FEATURE_WEIGHTS,
        "cut_low": P8_CUT_LOW,
        "cut_high": P8_CUT_HIGH,
        "recommended_for_production": P8_RECOMMENDED_FOR_PRODUCTION,
        "usage": "Verbatim reuse -- no fresh state-scoring fit. Notebook 51 applies this exact fitted "
                 "scorer to every real statement, then trains a fresh classifier on top to predict the "
                 "individual-level cure/no-cure outcome.",
    },
    "kpi_targets": COLLECTIONS_KPI_TARGETS,
    "ead_per_account_usd": EAD_PER_ACCOUNT_USD,
    "lgd_assumption": LGD_ASSUMPTION,
    "random_seed": RANDOM_SEED,
    "warp_resource_cap": {
        "cpu_fraction_cap": _PHASE4_CPU_FRACTION_CAP,
        "ram_fraction_cap": _PHASE4_RAM_FRACTION_CAP,
        "warp_thread_count": WARP_THREAD_COUNT,
        "max_ram_bytes": MAX_RAM_BYTES,
        "note": "Phase 4 tightened cap (92%/92%), min'd against the platform's original 95%/90% split -- "
                "see Section 2.",
    },
}
policy_path = COLLECTIONS_POLICY_DIR / "collections_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(COLLECTIONS_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("COLLECTIONS_ELIGIBLE_STATES is a non-empty subset of Problem 8's real states",
                              len(COLLECTIONS_ELIGIBLE_STATES) > 0
                              and set(COLLECTIONS_ELIGIBLE_STATES).issubset(set(P8_STATE_NAMES)))
_all_checks_passed &= _check("Low Severity correctly excluded from the eligible-for-collections state list",
                              "Low Severity" not in COLLECTIONS_ELIGIBLE_STATES)
_all_checks_passed &= _check("MIN_STATEMENTS_FOR_CURE_LABEL is the true minimum for 1 observed outcome (>= 2)",
                              MIN_STATEMENTS_FOR_CURE_LABEL >= 2)
_all_checks_passed &= _check("Cure-label upper-bound coverage is a real measured percentage in (0, 100]",
                              0 < CURE_LABEL_UPPER_BOUND_COVERAGE_PCT <= 100)
_all_checks_passed &= _check("Statement count stats are internally consistent (min <= p25 <= max)",
                              STATEMENT_COUNT_STATS["min"] <= STATEMENT_COUNT_STATS["p25"] <= STATEMENT_COUNT_STATS["max"])
_all_checks_passed &= _check("Reused Problem 1's real champion AUC (not fabricated)",
                              FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"))
_all_checks_passed &= _check("Reused Problem 8's real monitored feature universe verbatim (no duplicates)",
                              len(P8_MONITORED_FEATURES) == len(set(P8_MONITORED_FEATURES)))
_all_checks_passed &= _check("EAD/LGD were inherited from Notebook 08, not re-guessed",
                              EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
                              and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])
_all_checks_passed &= _check("Treatment-tier policy has exactly 3 named tiers",
                              len(COLLECTIONS_KPI_TARGETS["treatment_tier_policy"]["tiers"]) == 3)
_all_checks_passed &= _check("WARP thread count never exceeds the historical config's own value",
                              WARP_THREAD_COUNT <= _historical_thread_count)
_all_checks_passed &= _check("WARP RAM ceiling never exceeds the historical config's own value",
                              MAX_RAM_BYTES <= _historical_max_ram_bytes)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n\u2705 Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 50 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 50 Summary Artifact")

NB50_SUMMARY = {
    "notebook": "50_collections_optimization_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "collections_eligible_states": COLLECTIONS_ELIGIBLE_STATES,
    "min_statements_for_cure_label": MIN_STATEMENTS_FOR_CURE_LABEL,
    "cure_label_upper_bound_coverage_pct": CURE_LABEL_UPPER_BOUND_COVERAGE_PCT,
    "min_propensity_model_roc_auc_target": COLLECTIONS_KPI_TARGETS["min_propensity_model_roc_auc"],
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB50_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_50_summary.json"
with open(NB50_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB50_SUMMARY, f, indent=2)
print(f"Wrote: {NB50_SUMMARY_PATH}")

_section("NOTEBOOK 50 COMPLETE")
print(f"COLLECTIONS_ELIGIBLE_STATES (ASSUMPTION, reused from Problem 8)   : {COLLECTIONS_ELIGIBLE_STATES}")
print(f"CURE_DEFINITION (ASSUMPTION)                                     : {CURE_DEFINITION}")
print(f"min_propensity_model_roc_auc target (ASSUMPTION)                 : "
      f"{COLLECTIONS_KPI_TARGETS['min_propensity_model_roc_auc']}")
print(f"Cure-label upper-bound coverage (real)                           : "
      f"{CURE_LABEL_UPPER_BOUND_COVERAGE_PCT:.1f}%")
print(f"WARP cap this notebook forward (Phase 4 tightened)               : "
      f"{WARP_THREAD_COUNT} threads / {MAX_RAM_BYTES / 1e9:.1f} GB RAM")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 51_collections_optimization_modeling.ipynb -- scores every real statement with Problem 8's "
    "real fitted state formula, builds the real (current_state, next_state) cure/no-cure label, trains "
    "the real propensity-to-cure classifier on the collections-eligible population, reports the full "
    "classification metrics suite against the KPI target set here, and applies the treatment-tier policy "
    "to produce a ranked, real collections worklist."
)
